# Inference diagnostics

This notebook reads saved inference CSVs and prompt metadata only; it does not run model inference or refit the surface. The figures follow the controlled question behind each corpus instead of displaying every saved value as an interchangeable bar.

`arc_length_parallel` is the proposed stakes readout used here. The separately elicited rating study is a self-contained experiment and is not loaded or plotted in this notebook. Reference slot lists are never treated as calibrated or evenly spaced severity labels: matched analyses join exact family and slot text, and directional analyses use only explicit experimental poles or contrasts.

In [ ]:
MODEL = 'gemma-4-31B-it'  # Model directory name; available models are printed below.
ARTIFACTS_ROOT = None  # Optional path to the directory containing model folders.
FAMILIES = None  # Optional exact-family filter for reference, flipped, pairwise, and null analyses.
SAVE_HTML = False  # Save figures under notebooks/inference_figures/<MODEL>/.
MIN_CORRELATION_N = 4
OUTCOMES = ["arc_length_parallel"]

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/corpora/inference").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository root or notebooks directory.")
search_root = Path(ARTIFACTS_ROOT).expanduser().resolve() if ARTIFACTS_ROOT else ROOT / "artifacts"
locations = sorted(p for p in search_root.rglob("inference")
                   if p.is_dir() and (p / "severity/severity_arc_lengths.csv").exists())
available_models = sorted({p.parent.name for p in locations})
print("Available models:", available_models)
matches = [p for p in locations if p.parent.name == MODEL]
if len(matches) != 1:
    raise ValueError(f"Expected one inference directory for {MODEL!r}; found {matches}. Set ARTIFACTS_ROOT to disambiguate.")
INFERENCE = matches[0]
print("Reading:", INFERENCE)

LABELS = {
    "arc_length_parallel": "parallel coordinate",
}
POSITION_COLORS = {"before": "#0072B2", "after": "#D55E00"}
px.defaults.template = "plotly_white"
figures = []

def show(fig, slug):
    fig.update_layout(
        title_x=0.02, margin=dict(l=70, r=30, t=100, b=70),
        legend_title_text="", hoverlabel=dict(align="left"),
    )
    figures.append((slug, fig))
    fig.show()
    if SAVE_HTML:
        out = ROOT / "notebooks/inference_figures" / MODEL
        out.mkdir(parents=True, exist_ok=True)
        fig.write_html(out / f"{len(figures):02d}_{slug}.html", include_plotlyjs=True)

def add_zero_line(fig, axis="x"):
    if axis == "x":
        fig.add_vline(x=0, line_width=1, line_dash="dot", line_color="#555", row="all", col="all")
    else:
        fig.add_hline(y=0, line_width=1, line_dash="dot", line_color="#555", row="all", col="all")

def spearman_rows(frame, group_cols, x, outcomes, **constants):
    rows = []
    by = group_cols[0] if len(group_cols) == 1 else group_cols
    for key, group in frame.groupby(by, sort=False, dropna=False):
        key = (key,) if len(group_cols) == 1 else key
        identity = dict(zip(group_cols, key))
        for outcome in outcomes:
            valid = group[[x, outcome]].dropna()
            if len(valid) < MIN_CORRELATION_N or valid[x].nunique() < 2 or valid[outcome].nunique() < 2:
                rho = np.nan
            else:
                rho = valid[x].corr(valid[outcome], method="spearman")
            rows.append({**identity, **constants, "outcome": outcome, "rho": rho, "n": len(valid)})
    return pd.DataFrame(rows)


## Load and validate saved outputs

Prompt metadata is joined using exported identifying fields with one-to-one validation. Rating artifacts are intentionally ignored because they belong to a separate experiment. Missing optional datasets are reported; ambiguous prompt matches fail explicitly.

In [ ]:
data, inventory = {}, []
coordinate_columns = {"arc_length_parallel", "arc_length_orthogonal"}
for path in sorted(INFERENCE.glob("*/*_arc_lengths.csv")):
    name = path.parent.name
    df = pd.read_csv(path)
    prompt_path = path.parent / "prompts.json"
    if not prompt_path.exists():
        raise FileNotFoundError(f"Missing prompt metadata for {name}: {prompt_path}")
    records = json.loads(prompt_path.read_text(encoding="utf-8"))
    metadata = []
    for record in records:
        row = {k: v for k, v in record.items() if not isinstance(v, (dict, list))}
        row.update(record.get("template_metadata", {}))
        row.update(record.get("task_metadata", {}))
        row.update(task=record["task"], template_id=record["template_id"], prompt=record["text"])
        metadata.append(row)
    meta = pd.DataFrame(metadata)
    keys = [c for c in df.columns if c in meta.columns and c not in coordinate_columns]
    if not keys:
        raise ValueError(f"No identifying fields shared by CSV and prompt metadata for {name}")
    df = df.merge(meta, on=keys, how="left", validate="one_to_one", indicator=True, sort=False)
    if not df["_merge"].eq("both").all():
        raise ValueError(f"Unmatched saved prompts in {name}")
    df = df.drop(columns="_merge")
    if df["prompt"].duplicated().any():
        raise ValueError(f"Duplicate prompt text in {name}")
    if "family" in df:
        df["family"] = df["family"].fillna(df["template_id"])
    else:
        df["family"] = df["template_id"]
    df["dataset"] = name
    missing = {}
    for outcome in OUTCOMES:
        if outcome in df:
            df[outcome] = pd.to_numeric(df[outcome], errors="coerce").replace([np.inf, -np.inf], np.nan)
            missing[outcome] = int(df[outcome].isna().sum())
    data[name] = df
    inventory.append({"dataset": name, "rows": len(df), **{f"missing {LABELS[k]}": v for k, v in missing.items()}})

display(pd.DataFrame(inventory))
if "severity" not in data:
    raise FileNotFoundError("The selected model has no severity reference corpus.")
baseline = data["severity"].copy()
if FAMILIES is not None:
    baseline = baseline[baseline["family"].isin(FAMILIES)].copy()


## Reference corpus

The reference corpus supplies exact family-and-slot matches for the counterfactual controls below. It has no declared severity labels or calibrated slot ordering, so it is not given a standalone ordering plot.

In [ ]:
print(f"Reference rows available for exact matching: {len(baseline):,}")

## Matched counterfactual and vocabulary controls

These plots target the interventions directly. Flipped prompts resolve exactly one harm, so the association between their parallel coordinate and the exact matched reference coordinate should reverse if the readout follows residual stakes. Pairwise prompts put the reference-low and reference-high values in a setting designed to reverse their actual consequences; the plotted contrast is explicitly `high pole - low pole`, where negative supports reversal. Null prompts mention the same exact values in non-consequential string or definition tasks; correlations with the exact matched reference coordinate should collapse toward zero.

In [ ]:
reference_keys = baseline[["family", "severity_word", "arc_length_parallel"]].copy()
reference_keys = reference_keys.rename(columns={"arc_length_parallel": "reference_parallel"})

if "severity_flipped" in data:
    flipped = data["severity_flipped"].copy()
    if FAMILIES is not None:
        flipped = flipped[flipped["family"].isin(FAMILIES)]
    matched = flipped.merge(
        reference_keys,
        on=["family", "severity_word"], how="inner", validate="one_to_one",
    )
    flipped_rho = spearman_rows(
        matched, ["family"], "reference_parallel", ["arc_length_parallel"],
    )
    fig = px.scatter(
        flipped_rho, x="rho", y="family", hover_data=["n"],
        labels={"rho": "rho(reference parallel, resolved-harm parallel)", "family": "matched family"},
        title=f"{MODEL}: resolving one harm should reverse the within-family association",
        height=max(500, 24 * flipped_rho["family"].nunique()),
    )
    fig.update_xaxes(range=[-1.05, 1.05])
    add_zero_line(fig, "x")
    show(fig, "flipped_rank_reversal")

if "severity_pairwise" in data:
    pairwise = data["severity_pairwise"].copy()
    if FAMILIES is not None:
        pairwise = pairwise[pairwise["family"].isin(FAMILIES)]
    rows = []
    for outcome in [m for m in OUTCOMES if m in pairwise]:
        wide = pairwise.pivot(index=["task", "family"], columns="severity_pole", values=outcome)
        if {"low", "high"}.issubset(wide.columns):
            part = (wide["high"] - wide["low"]).rename("difference").reset_index()
            part["outcome"] = outcome
            rows.append(part)
    if rows:
        pair_effects = pd.concat(rows, ignore_index=True)
        pair_effects["outcome_label"] = pair_effects["outcome"].map(LABELS)
        fig = px.scatter(
            pair_effects, x="difference", y="task", color="family",
            facet_col="outcome_label", hover_data=["family"],
            labels={"difference": "high reference pole - low reference pole", "task": "reversal scenario"},
            title=f"{MODEL}: explicit pairwise reversals (negative is the predicted direction)", height=560,
        )
        fig.update_xaxes(matches=None)
        fig.for_each_annotation(lambda a: a.update(text=a.text.split("=", 1)[-1]))
        add_zero_line(fig, "x")
        show(fig, "pairwise_reversal_effects")

if "severity_null" in data:
    null = data["severity_null"].copy()
    if FAMILIES is not None:
        null = null[null["family"].isin(FAMILIES)]
    null = null.merge(
        reference_keys,
        on=["family", "severity_word"], how="inner", validate="many_to_one",
    )
    null_rho = spearman_rows(
        null, ["family", "frame"], "reference_parallel", ["arc_length_parallel"],
    ).rename(columns={"frame": "condition"})
    condition_order = ["letter_count", "title_case", "translation", "definition"]
    fig = px.box(
        null_rho, x="condition", y="rho", points="all",
        category_orders={"condition": condition_order}, hover_data=["family", "n"],
        labels={"rho": "rho(reference parallel, null-frame parallel)", "condition": "prompt frame"},
        title=f"{MODEL}: exact severity vocabulary should lose its ladder in null frames", height=520,
    )
    fig.update_yaxes(range=[-1.05, 1.05])
    fig.update_xaxes(tickangle=45)
    add_zero_line(fig, "y")
    show(fig, "null_vocabulary_control")


## Quantitative magnitude and numeral-format invariance

Magnitude prompts vary one quantity over several orders of magnitude. Trajectories are plotted against `log10(value)`, the regressor the corpus was designed around. Family-by-format slopes summarize quantitative response; exact numeric-versus-word differences expose numeral-format sensitivity that a slope alone could hide. The numeric savings prompts are also checked against byte-identical reference prompts as a data-integrity check.

In [ ]:
if "severity_magnitude" in data:
    magnitude = data["severity_magnitude"].sort_values(["family", "number_format", "log10_value"]).copy()
    magnitude_outcomes = [m for m in OUTCOMES if m in magnitude]
    long = magnitude.melt(
        id_vars=["family", "number_format", "log10_value", "value_text", "prompt"],
        value_vars=magnitude_outcomes, var_name="outcome", value_name="response",
    )
    long["outcome_label"] = long["outcome"].map(LABELS)
    fig = px.line(
        long, x="log10_value", y="response", color="number_format", markers=True,
        facet_row="outcome_label", facet_col="family", hover_data=["value_text", "prompt"],
        labels={"log10_value": "log10(quantity)", "response": "saved response", "family": "magnitude family"},
        title=f"{MODEL}: response to a fixed incident's quantitative magnitude", height=850,
    )
    fig.update_yaxes(matches=None)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=", 1)[-1]))
    show(fig, "magnitude_trajectories")

    fits = []
    for (family, number_format), group in magnitude.groupby(["family", "number_format"], sort=False):
        for outcome in magnitude_outcomes:
            valid = group[["log10_value", outcome]].dropna()
            if len(valid) < 3 or valid["log10_value"].nunique() < 2:
                continue
            slope, intercept = np.polyfit(valid["log10_value"], valid[outcome], 1)
            predicted = slope * valid["log10_value"].to_numpy() + intercept
            residual = ((valid[outcome].to_numpy() - predicted) ** 2).sum()
            total = ((valid[outcome].to_numpy() - valid[outcome].mean()) ** 2).sum()
            fits.append({"family": family, "number_format": number_format, "outcome": outcome,
                         "slope": slope, "r_squared": np.nan if total == 0 else 1 - residual / total, "n": len(valid)})
    fits = pd.DataFrame(fits)
    fits["outcome_label"] = fits["outcome"].map(LABELS)
    fig = px.scatter(
        fits, x="slope", y="family", color="number_format", symbol="number_format",
        facet_col="outcome_label", hover_data=["r_squared", "n"],
        labels={"slope": "change per ten-fold increase", "family": "magnitude family"},
        title=f"{MODEL}: fitted magnitude slopes (hover for R-squared)", height=480,
    )
    fig.update_xaxes(matches=None)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=", 1)[-1]))
    add_zero_line(fig, "x")
    show(fig, "magnitude_slopes")

    format_rows = []
    for outcome in magnitude_outcomes:
        wide = magnitude.pivot(index=["family", "value"], columns="number_format", values=outcome)
        if {"numeric", "words"}.issubset(wide.columns):
            part = (wide["words"] - wide["numeric"]).rename("format_difference").reset_index()
            part["outcome"] = outcome
            format_rows.append(part)
    if format_rows:
        format_effects = pd.concat(format_rows, ignore_index=True)
        format_effects["outcome_label"] = format_effects["outcome"].map(LABELS)
        fig = px.box(
            format_effects, x="family", y="format_difference", points="all",
            facet_col="outcome_label", hover_data=["value"],
            labels={"format_difference": "words - digits", "family": "magnitude family"},
            title=f"{MODEL}: exact-quantity numeral-format effects (zero means invariant)", height=500,
        )
        fig.update_yaxes(matches=None)
        fig.update_xaxes(tickangle=45)
        fig.for_each_annotation(lambda a: a.update(text=a.text.split("=", 1)[-1]))
        add_zero_line(fig, "y")
        show(fig, "magnitude_format_effects")

    repeated = magnitude[magnitude["number_format"].eq("numeric")].merge(
        baseline[["prompt", "arc_length_parallel"]],
        on="prompt", suffixes=("_magnitude", "_reference"), validate="one_to_one",
    )
    if not repeated.empty:
        checks = []
        for coordinate in ["arc_length_parallel"]:
            delta = (repeated[f"{coordinate}_magnitude"] - repeated[f"{coordinate}_reference"]).abs()
            checks.append({"coordinate": LABELS[coordinate], "matched prompts": len(delta), "max absolute difference": delta.max()})
        display(pd.DataFrame(checks).round({"max absolute difference": 6}))


## Composition: adding a second harm

Each composition receives its own figure: low-low, low-high, high-low, and high-high. For asymmetric pairs, `a then b` is low-high and `b then a` is high-low. The left panel shows the two component harms, the same-first-clause inert control, and the conjunction. The right panel shows `conjunction - larger single` and `conjunction - matched inert control`; negative values in the first contrast are the sharp averaging failure. Both orders remain visible for the symmetric compositions because final-token activations may be order-sensitive.

In [ ]:
if "severity_composition" in data:
    composition = data["severity_composition"].copy()
    required_roles = {"a", "b", "a_and_b", "b_and_a", "a_and_inert", "b_and_inert"}
    order_specs = [
        ("a then b", "a", "b", "a_and_b", "a_and_inert"),
        ("b then a", "b", "a", "b_and_a", "b_and_inert"),
    ]
    pole_labels = {"low_low": {"a": "low", "b": "low"},
                   "low_high": {"a": "low", "b": "high"},
                   "high_high": {"a": "high", "b": "high"}}
    profile_rows = []
    for (task, pair_type), group in composition.groupby(["task", "pair_type"], sort=False):
        indexed = group.set_index("role")
        values = indexed["arc_length_parallel"]
        if not required_roles.issubset(values.index):
            raise ValueError(f"{task}: incomplete composition roles")
        poles = pole_labels[pair_type]
        higher_single = max(values["a"], values["b"])
        for order, first, second, conjunction, inert_control in order_specs:
            composition_type = f"{poles[first]}-{poles[second]}"
            series = f"{task}: {order}"
            for item, value in [
                ("first harm alone", values[first]),
                ("second harm alone", values[second]),
                ("first harm + inert", values[inert_control]),
                ("both harms", values[conjunction]),
            ]:
                profile_rows.append({"composition": composition_type, "task": task, "order": order,
                                     "series": series, "panel": "coordinate profile",
                                     "item": item, "value": value})
            for item, value in [
                ("both - larger single", values[conjunction] - higher_single),
                ("both - inert control", values[conjunction] - values[inert_control]),
            ]:
                profile_rows.append({"composition": composition_type, "task": task, "order": order,
                                     "series": series, "panel": "diagnostic contrasts",
                                     "item": item, "value": value})

    profiles = pd.DataFrame(profile_rows)
    composition_order = ["low-low", "low-high", "high-low", "high-high"]
    task_order = composition["task"].drop_duplicates().tolist()
    task_colors = {task: px.colors.qualitative.Safe[i % len(px.colors.qualitative.Safe)]
                   for i, task in enumerate(task_order)}
    dash_by_order = {"a then b": "solid", "b then a": "dash"}
    for composition_type in composition_order:
        part = profiles[profiles["composition"].eq(composition_type)]
        if part.empty:
            print(f"No saved rows for composition {composition_type}")
            continue
        fig = make_subplots(rows=1, cols=2, subplot_titles=["parallel-coordinate profile", "diagnostic contrasts"])
        for series, series_rows in part.groupby("series", sort=False):
            task = series_rows["task"].iloc[0]
            order = series_rows["order"].iloc[0]
            line = dict(color=task_colors[task], width=2, dash=dash_by_order[order])
            raw = series_rows[series_rows["panel"].eq("coordinate profile")]
            diagnostic = series_rows[series_rows["panel"].eq("diagnostic contrasts")]
            fig.add_trace(go.Scatter(
                x=raw["item"], y=raw["value"], mode="lines+markers", name=series,
                legendgroup=series, line=line, marker=dict(size=8),
                hovertemplate="%{x}<br>parallel=%{y:.3g}<extra>" + series + "</extra>",
            ), row=1, col=1)
            fig.add_trace(go.Scatter(
                x=diagnostic["item"], y=diagnostic["value"], mode="lines+markers", name=series,
                legendgroup=series, showlegend=False, line=line, marker=dict(size=8),
                hovertemplate="%{x}<br>difference=%{y:.3g}<extra>" + series + "</extra>",
            ), row=1, col=2)
        fig.add_hline(y=0, line_width=1, line_dash="dot", line_color="#555", row=1, col=2)
        fig.update_xaxes(type="category", tickangle=45, row=1, col=1)
        fig.update_xaxes(type="category", tickangle=45, row=1, col=2)
        fig.update_yaxes(title_text="parallel coordinate", row=1, col=1)
        fig.update_yaxes(title_text="parallel-coordinate difference", row=1, col=2)
        fig.update_layout(title=f"{MODEL}: {composition_type} composition", height=520)
        show(fig, "composition_" + composition_type.replace("-", "_"))


## Wording robustness

These prompts keep the requested task fixed and vary only wording and expressed distress. Because the variants have no declared severity rank, the plot does not connect them in saved order. It shows the within-task response range across all variants; smaller ranges indicate greater wording invariance. The table separates average within-task variance from variance between task means.

In [ ]:
if "severity_wording" in data:
    wording = data["severity_wording"].copy()
    wording_outcomes = [m for m in OUTCOMES if m in wording]
    ranges = wording.groupby("task", sort=False)[wording_outcomes].agg(lambda s: s.max() - s.min()).reset_index()
    long_ranges = ranges.melt(id_vars="task", var_name="outcome", value_name="within_task_range")
    long_ranges["outcome_label"] = long_ranges["outcome"].map(LABELS)
    parallel_order = ranges.sort_values("arc_length_parallel")["task"].tolist() if "arc_length_parallel" in ranges else ranges["task"].tolist()
    fig = px.scatter(
        long_ranges, x="within_task_range", y="task", facet_col="outcome_label",
        category_orders={"task": parallel_order},
        labels={"within_task_range": "max - min across wording variants", "task": "fixed requested task"},
        title=f"{MODEL}: sensitivity to wording and expressed distress", height=650,
    )
    fig.update_xaxes(matches=None)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=", 1)[-1]))
    show(fig, "wording_within_task_ranges")

    variance_rows = []
    for outcome in wording_outcomes:
        within = wording.groupby("task")[outcome].var(ddof=1).mean()
        between = wording.groupby("task")[outcome].mean().var(ddof=1)
        variance_rows.append({"outcome": LABELS[outcome], "mean within-task variance": within,
                              "between-task-mean variance": between,
                              "within share": within / (within + between) if within + between > 0 else np.nan})
    display(pd.DataFrame(variance_rows).round({"mean within-task variance": 4,
                                               "between-task-mean variance": 4,
                                               "within share": 4}))


## Context ladder, controls, and placement

For every task, each framed response is averaged over three interchangeable phrasings and then differenced from that task's bare scenario. The ladder runs from a live answer that will be acted on immediately to an impossible fictional world. Thin lines are individual tasks; thick lines are task means with standard-error bars. The filler-minus-bare plot measures the effect of adding a length-matched but consequence-neutral sentence. The final plot isolates placement by subtracting the before-framing mean from the after-framing mean for the same task and context.

In [ ]:
if "context" in data:
    context = data["context"].copy()
    context_outcomes = [m for m in OUTCOMES if m in context]
    bare = context[context["context"].eq("none")].copy()
    if bare["task"].duplicated().any():
        raise ValueError("Expected exactly one bare context control per task.")
    framed = context[~context["context"].eq("none")].copy()
    means = framed.groupby(
        ["task", "context", "context_position", "ladder_rank", "is_control"],
        sort=False, dropna=False,
    )[context_outcomes].mean().reset_index()
    bare_values = bare.set_index("task")[context_outcomes]
    for outcome in context_outcomes:
        means[f"delta_{outcome}"] = means[outcome] - means["task"].map(bare_values[outcome])

    ladder = means[~means["is_control"].astype(bool)].copy()
    frame_order = (ladder.dropna(subset=["ladder_rank"])
                   .sort_values("ladder_rank")[["ladder_rank", "context"]].drop_duplicates())
    subplot_titles = [LABELS[m] for m in context_outcomes]
    fig = make_subplots(rows=1, cols=len(context_outcomes), subplot_titles=subplot_titles)
    for col, outcome in enumerate(context_outcomes, start=1):
        delta_col = f"delta_{outcome}"
        for position, color in POSITION_COLORS.items():
            positioned = ladder[ladder["context_position"].eq(position)]
            for _, task_rows in positioned.groupby("task", sort=False):
                task_rows = task_rows.sort_values("ladder_rank")
                fig.add_trace(go.Scatter(
                    x=task_rows["ladder_rank"], y=task_rows[delta_col], mode="lines",
                    line=dict(color=color, width=1), opacity=0.20, showlegend=False,
                    hovertemplate="task=%{customdata[0]}<br>context=%{customdata[1]}<br>delta=%{y:.3g}<extra></extra>",
                    customdata=task_rows[["task", "context"]],
                ), row=1, col=col)
            summary = positioned.groupby("ladder_rank", sort=True)[delta_col].agg(["mean", "std", "count"]).reset_index()
            summary["sem"] = summary["std"] / np.sqrt(summary["count"])
            fig.add_trace(go.Scatter(
                x=summary["ladder_rank"], y=summary["mean"], mode="lines+markers", name=position,
                legendgroup=position, showlegend=(col == 1), line=dict(color=color, width=4),
                marker=dict(size=8), error_y=dict(type="data", array=summary["sem"], visible=True),
                hovertemplate=position + "<br>mean delta=%{y:.3g}<extra></extra>",
            ), row=1, col=col)
        fig.update_xaxes(
            tickmode="array", tickvals=frame_order["ladder_rank"], ticktext=frame_order["context"],
            tickangle=45, title_text="consequentiality ladder", row=1, col=col,
        )
        fig.update_yaxes(title_text="framed - bare", row=1, col=col)
    fig.update_layout(title=f"{MODEL}: matched context effect across the consequentiality ladder", height=560)
    show(fig, "context_ladder_effects")

    filler = means[means["context"].eq("filler")].copy()
    if not filler.empty:
        filler_long = filler.melt(
            id_vars=["task", "context_position"],
            value_vars=[f"delta_{m}" for m in context_outcomes],
            var_name="outcome", value_name="difference",
        )
        filler_long["outcome"] = filler_long["outcome"].str.removeprefix("delta_")
        filler_long["outcome_label"] = filler_long["outcome"].map(LABELS)
        fig = px.scatter(
            filler_long, x="difference", y="task", color="context_position",
            facet_col="outcome_label", color_discrete_map=POSITION_COLORS,
            labels={"difference": "length-matched filler - bare", "task": "matched task"},
            title=f"{MODEL}: extra-sentence control", height=440,
        )
        fig.update_xaxes(matches=None)
        fig.for_each_annotation(lambda a: a.update(text=a.text.split("=", 1)[-1]))
        add_zero_line(fig, "x")
        show(fig, "context_filler_control")

    placement_rows = []
    for outcome in context_outcomes:
        wide = means.pivot(index=["task", "context", "ladder_rank", "is_control"],
                           columns="context_position", values=outcome)
        if {"before", "after"}.issubset(wide.columns):
            part = (wide["after"] - wide["before"]).rename("placement_effect").reset_index()
            part["outcome"] = outcome
            placement_rows.append(part)
    if placement_rows:
        placement = pd.concat(placement_rows, ignore_index=True)
        placement["outcome_label"] = placement["outcome"].map(LABELS)
        context_order = frame_order["context"].tolist() + ["filler"]
        fig = px.strip(
            placement, x="context", y="placement_effect", color="task",
            facet_col="outcome_label", category_orders={"context": context_order},
            labels={"placement_effect": "after - before", "context": "framing condition"},
            title=f"{MODEL}: framing placement sensitivity", height=500,
        )
        fig.update_yaxes(matches=None)
        fig.update_xaxes(tickangle=45)
        fig.for_each_annotation(lambda a: a.update(text=a.text.split("=", 1)[-1]))
        add_zero_line(fig, "y")
        show(fig, "context_placement_effects")

    phrasing_sd = framed.groupby(["task", "context", "context_position"], sort=False)[context_outcomes].std(ddof=1)
    reliability = phrasing_sd.median().rename("median within-cell SD across phrasings").reset_index()
    reliability["outcome"] = reliability["index"].map(LABELS)
    display(reliability[["outcome", "median within-cell SD across phrasings"]].round(4))


In [ ]:
print(f"Created {len(figures)} hypothesis-directed Plotly figures from {INFERENCE}")
if SAVE_HTML:
    print("Saved HTML figures to:", ROOT / "notebooks/inference_figures" / MODEL)